# 10 — Aggregation Functions (groupBy, agg, count / sum / avg / min / max)

Aggregation functions on an employees DataFrame (13 rows, 3 depts, deliberate duplicates + injected NULLs) — one operation per cell, each DataFrame API example paired with its Spark SQL equivalent:

* `df.count()` (ACTION) vs `F.count()` (TRANSFORMATION) — NULL-skipping, `F.count(lit(1))` / `COUNT(*)`
* NULL behaviour of aggregates — `sum`/`avg`/`min`/`max` skip NULLs; `avg()` divides by the NON-NULL count; all-NULL column → NULL (count → 0)
* Handling NULLs — `coalesce`, `ifnull`, `nvl`, `nullif`, `isNull()`/`isNotNull()`, `df.na.fill()`, `df.na.drop()`
* One all-NULL row — counted by `COUNT(*)`, skipped by value aggregates, forms its own null group in `groupBy`, excluded from `countDistinct`, collapsed by `dropDuplicates`, removed by `na.drop(how="all")`
* Whole-DataFrame aggregations — `sum()`, `avg()`, `min()`, `max()` via `select()`
* `groupBy()` — per-group counts, `agg()` with multiple named aggregations, multi-column grouping, `countDistinct()`
* Sorting aggregations + `filter()`-after-groupBy as the `HAVING` equivalent
* ⚠️ Duplicates gotcha — `sum()` before vs after `dropDuplicates()` (820000 vs 560000)
* Quick Revision table + handwritten-style revision notes at the end

## Built-in Aggregation Functions — Quick Reference

Every one of these skips NULLs (except `COUNT(*)`), and all can run whole-DataFrame or per `groupBy()` group.

| Function (DF API) | SQL equivalent | What it does |
| --- | --- | --- |
| `F.count(col)` | `COUNT(col)` | Number of NON-NULL values |
| `F.count(F.lit(1))` / `F.count("*")` | `COUNT(*)` | Number of ROWS — NULLs included (the two DF forms are equivalent) |
| `F.countDistinct(col)` / `F.count_distinct(col)` | `COUNT(DISTINCT col)` | Number of UNIQUE non-NULL values |
| `F.sum(col)` | `SUM(col)` | Total of non-NULL values |
| `F.sumDistinct(col)` / `F.sum_distinct(col)` | `SUM(DISTINCT col)` | Total of UNIQUE values only (each value once) |
| `F.avg(col)` | `AVG(col)` | Mean — sum ÷ NON-NULL count |
| `F.min(col)` | `MIN(col)` | Smallest non-NULL value |
| `F.max(col)` | `MAX(col)` | Largest non-NULL value |
| `F.first(col, ignorenulls=True)` | `FIRST(col) [IGNORE NULLS]` | First value in the group — usage: `F.first(col("salary"), ignorenulls=True)` → first NON-NULL salary (IGNORE NULLS); `F.first(col("salary"))` → may return NULL (RESPECT NULLS, the default) |
| `F.last(col, ignorenulls=True)` | `LAST(col) [IGNORE NULLS]` | Last value in the group — usage: `F.last(col("salary"), True)` → last NON-NULL salary (IGNORE NULLS); `F.last(col("salary"))` → may return NULL (RESPECT NULLS, the default) |
| `F.collect_list(col)` | `collect_list(col)` / `ARRAY_AGG` | Array of ALL values — duplicates kept |
| `F.collect_set(col)` | `collect_set(col)` | Array of UNIQUE values — duplicates dropped |
| `F.stddev(col)` | `STDDEV(col)` | Sample standard deviation |
| `F.stddev_pop(col)` / `F.stddev_samp(col)` | `STDDEV_POP` / `STDDEV_SAMP` | Population / sample standard deviation |
| `F.variance(col)` | `VARIANCE(col)` | Sample variance (stddev²) |
| `F.var_pop(col)` / `F.var_samp(col)` | `VAR_POP` / `VAR_SAMP` | Population / sample variance |
| `F.skewness(col)` | `SKEWNESS(col)` | Asymmetry of the distribution (0 = symmetric) |
| `F.kurtosis(col)` | `KURTOSIS(col)` | Heavy-tailedness of the distribution |
| `F.corr(col1, col2)` | `CORR(col1, col2)` | Pearson correlation between two columns (−1 to +1) |
| `F.covar_pop(col1, col2)` / `F.covar_samp(col1, col2)` | `COVAR_POP` / `COVAR_SAMP` | Population / sample covariance of two columns |
| `F.approx_count_distinct(col)` | `APPROX_COUNT_DISTINCT(col)` | FAST approximate distinct count (HyperLogLog++) |
| `F.percentile_approx(col, p)` | `PERCENTILE_APPROX(col, p)` | Approximate p-th percentile (median = p 0.5) |
| `F.grouping(col)` | `GROUPING(col)` | 1 when the NULL is from a CUBE/ROLLUP super-row, else 0 |
| `F.grouping_id(col1, col2)` | `GROUPING_ID(...)` | Bitmask of which columns were grouped in a CUBE/ROLLUP row |

SQL-only extras: `MEDIAN(col)` (exact 50th percentile), `MODE(col)` (most frequent value), `ANY_VALUE(col)` (any one value — lets you SELECT a non-grouped column), `EVERY / SOME / ANY` (boolean tests over the group), `BIT_AND / BIT_OR / BIT_XOR` (bitwise aggregates).

**IGNORE NULLS / RESPECT NULLS** — only a few functions take this clause: `first`, `last`, `any_value` (plus the window functions `lag`, `lead`, `nth_value`).

* `RESPECT NULLS` is the DEFAULT — `F.first(col)` / `FIRST(col)` can return NULL when the first row's value is NULL
* `IGNORE NULLS` skips them — `F.first(col, ignorenulls=True)` / `FIRST(col) IGNORE NULLS` returns the first NON-NULL value instead
* `F.any_value(col, ignoreNulls=True)` takes the same flag (SQL: `ANY_VALUE(col) IGNORE NULLS`)
* Everything else in this table (`sum`, `avg`, `min`, `max`, `count`, `collect_list`, `collect_set`) skips NULLs automatically — the clause does not exist for them

> [!TIP]
> 💡 The core ones — `count`, `sum`, `avg`, `min`, `max`, `countDistinct` — all have worked examples later in this notebook.

In [0]:
# Setup: SparkSession + employees DataFrame (dept column for groupBy; duplicates by design)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit

spark = SparkSession.builder.appName("aggregation-functions").getOrCreate()

# total 13 rows, including 2 duplicates
emp_data = [
    (10, "Anil", "IT", 50000, 18),
    (11, "Vikas", "HR", 75000, 16),
    (12, "Nisha", "IT", 40000, 18),
    (13, "Nidhi", "SALES", 60000, 17),
    (14, "Priya", "HR", 80000, 18),
    (15, "Mohit", "IT", 45000, 18),
    (16, "Rajesh", "SALES", 90000, 10),
    (17, "Raman", "HR", 55000, 16),
    (18, "Sam", "SALES", 65000, 17),
    (15, "Mohit", "IT", 45000, 18),    # exact duplicate
    (13, "Nidhi", "SALES", 60000, 17), # exact duplicate
    (14, "Priya", "HR", 90000, 18),    # same id, different salary
    (18, "Sam", "SALES", 65000, 17),   # exact duplicate
]
print(f"{len(emp_data) = }")

schema = """
    id INT,
    name STRING,
    dept STRING,
    salary DOUBLE,
    mgr_id INT
"""
employees_df = spark.createDataFrame(emp_data, schema=schema)

employees_df.show()

## The Dataset — duplicates by design (intentionally)

---

13 rows, 9 unique employees, 3 departments (IT / HR / SALES). The duplicates are deliberate — they power the sum-before-vs-after-dedup gotcha at the end.

In [0]:
# Total Count including duplicates
print("Count:", employees_df.count())

employees_df.show()

## `df.count()` (action) vs `F.count()` (transformation) — with NULLs

---

Two DIFFERENT counts: `df.count()` is an **ACTION** — runs immediately and returns a number. `F.count(col(...))` is a **TRANSFORMATION** — it produces an aggregate column, and it **skips NULLs** (like SQL `COUNT(col)`).
`F.count(lit(1))` / `F.count("*")` is the `COUNT(*)` equivalent — every row counts, NULLs included.

In [0]:
# df.count() — ACTION: returns a NUMBER (not a DataFrame) and executes immediately
total_rows = employees_df.count()

print("df.count() returns:", total_rows, "—", type(total_rows).__name__)

In [0]:
# Inject NULL salaries (ids 15 and 17) — the two counts behave differently around NULLs
employees_null_df = employees_df.withColumn(
    "salary",
    F.when(col("id").isin(15, 17), F.lit(None)).otherwise(col("salary")),
)

employees_null_df.sort(col("id")).show()

In [0]:
# df.count() = ROWS (NULLs included); F.count(col) = NON-NULL values only
row_count = employees_null_df.count()

print("df.count() rows (NULLs included):", row_count)

employees_null_df.select(
    F.count(col("salary")).alias("count_salary_skips_nulls"),
    F.count(col("id")).alias("count_id_non_null"),
).show()

In [0]:
# F.count(lit(1)) / F.count("*") — the COUNT(*) equivalent: every ROW, NULLs included
employees_null_df.select(
    F.count(F.lit(1)).alias("count_lit_1"),
    F.count("*").alias("count_star"),
).show()

In [0]:
# Spark SQL Equivalent — COUNT(*) counts rows; COUNT(col) skips NULLs
employees_null_df.createOrReplaceTempView("employees_null")

spark.sql("""
    SELECT COUNT(*)     AS count_star_rows,
           COUNT(salary) AS count_salary_no_nulls
    FROM employees_null
""").show()

## NULL behaviour — `sum()` / `avg()` / `min()` / `max()` ignore NULLs

---

Every aggregate function SKIPS NULLs — `sum()` adds only non-NULL values, `avg()` divides by the NON-NULL count only (NOT the row count), and `min()`/`max()` never return NULL when any value exists.
When ALL values in a column are NULL: `sum`/`avg`/`min`/`max` return NULL — only `count()` returns a number (0).

In [0]:
# sum/avg/min/max all SKIP NULLs — avg divides by the NON-NULL count (10), not 13 rows
employees_null_df.select(
    F.count(col("salary")).alias("n_non_null"),
    F.sum(col("salary")).alias("sum_salary"),
    F.avg(col("salary")).alias("avg_salary"),
    F.min(col("salary")).alias("min_salary"),
    F.max(col("salary")).alias("max_salary"),
).show()

In [0]:
# Spark SQL Equivalent — every aggregate skips NULLs the same way
spark.sql("""
    SELECT COUNT(salary) AS n_non_null,
           SUM(salary)   AS sum_salary,
           AVG(salary)   AS avg_salary,
           MIN(salary)   AS min_salary,
           MAX(salary)   AS max_salary
    FROM employees_null
""").show()

In [0]:
# When ALL values are NULL: sum/avg/min/max return NULL — only count() returns a number (0)
all_null_df = employees_null_df.select(F.lit(None).cast("double").alias("null_salary"))

all_null_df.select(
    F.count(col("null_salary")).alias("count_returns_0"),
    F.sum(col("null_salary")).alias("sum_is_null"),
    F.avg(col("null_salary")).alias("avg_is_null"),
    F.min(col("null_salary")).alias("min_is_null"),
    F.max(col("null_salary")).alias("max_is_null"),
).show()

## Handling NULLs — `coalesce`, `IFNULL`, `NVL`, `NULLIF`, `na.fill` / `na.drop`

---

| Function | Where | What it returns |
| --- | --- | --- |
| `F.coalesce(c1, c2, ...)` / `COALESCE` | DF + SQL | FIRST non-NULL argument |
| `IFNULL(c, d)` | SQL only | `d` if `c` is NULL, else `c` (a 2-arg COALESCE) |
| `NVL(c, d)` | SQL only | alias of `IFNULL` — identical behaviour |
| `NULLIF(a, b)` | SQL only | NULL if `a == b`, else `a` (DF: `when`/`otherwise`) |
| DF: `col.isNull()` / `col.isNotNull()`<br>SQL: `IS NULL` / `IS NOT NULL` | DF + SQL | NULL tests |
| `df.na.fill()` / `fillna()` | DF only | replace NULLs (dict per column) |
| `df.na.drop()` / `dropna()` | DF only | drop rows containing NULLs (subset option) |

> [!WARNING]
> ⚠️ `col("salary") == NULL` NEVER matches — comparisons with NULL return NULL. Use `isNull()` / `IS NULL`.

**Which one to use, and when?**

* **DF API → `F.coalesce` always** — `IFNULL`/`NVL` don't exist in PySpark
* **SQL, 2+ fallbacks → `COALESCE(c1, c2, ...)`** — takes any number of arguments
* **SQL, one simple default → `IFNULL(c, d)` or `NVL(c, d)`** — exactly 2 arguments, identical behaviour (`NVL` is the Oracle-style alias, `IFNULL` the SQL-Server style); pick one and use it consistently

> [!TIP]
> 💡 There is NO performance difference — Spark compiles `IFNULL`/`NVL` to the same expression as a 2-arg `coalesce`.

> [!TIP]
> 💡 Default to `coalesce`: it works in BOTH the DF API and SQL, and it covers the 2-arg case too. Reach for `IFNULL`/`NVL` only in SQL for readability of a single fallback.

In [0]:
# coalesce() — FIRST non-NULL argument: replaces NULL salaries with a 50000 default
(
    employees_null_df
    .select(
        col("name"),
        col("salary"),
        F.coalesce(col("salary"), F.lit(50000)).alias("salary_filled"),
    )
    .sort(col("id"))
).show()

In [0]:
# Spark SQL — ifnull() and nvl() are 2-argument NULL-replacers; COALESCE is the general form
spark.sql("""
    SELECT name, salary,
           COALESCE(salary, 50000) AS via_coalesce,
           IFNULL(salary, 50000)   AS via_ifnull,
           NVL(salary, 50000)     AS via_nvl
    FROM employees_null
    WHERE salary IS NULL
""").show()

In [0]:
# nullif(a, b) — returns NULL when a == b (neutralizes a sentinel value), else a
# DF API equivalent: F.when(col("salary") == 50000, F.lit(None)).otherwise(col("salary"))
spark.sql("""
    SELECT name, salary,
           nullif(salary, 50000) AS salary_nullif_50k
    FROM employees_null
    WHERE salary IS NOT NULL
    ORDER BY id
""").show()

In [0]:
# isNull() / isNotNull() — SQL: IS NULL / IS NOT NULL (never compare with == NULL)
print("NULL salaries     :", employees_null_df.filter(col("salary").isNull()).count())
print("Non-NULL salaries :", employees_null_df.filter(col("salary").isNotNull()).count())

In [0]:
# df.na.fill() / fillna() — DataFrame-level NULL replacement (dict: one default per column)
filled_df = employees_null_df.na.fill({"salary": 50000})

print("NULL salaries after fill:", filled_df.filter(col("salary").isNull()).count())
print("Sum after fill          :", filled_df.agg(F.sum(col("salary"))).first()[0])

In [0]:
# df.na.drop() / dropna() — remove rows containing NULLs (subset= targets specific columns)
kept_df = employees_null_df.na.drop(subset=["salary"])

print("Rows before drop:", employees_null_df.count())
print("Rows after drop :", kept_df.count())

## One Entire NULL Row — how everything treats it

---

A row where EVERY column is NULL (id, name, dept, salary, mgr_id). Classic scenarios:

| Operation | Result on the all-NULL row |
| --- | --- |
| `df.count()` / `COUNT(*)` | COUNTED — 14 rows |
| `F.count(col)` / `sum` / `avg` / `min` / `max` | SKIPPED — NULL is not a value |
| `groupBy()` / `GROUP BY` | NULLs group TOGETHER — a `null` group appears |
| `countDistinct` / `COUNT(DISTINCT)` | EXCLUDED — NULL is not a value |
| `dropDuplicates(["id"])` | NULLs treated as EQUAL — one NULL row survives |
| `df.na.drop(how="all")` | REMOVES fully-NULL rows |

> [!WARNING]
> ⚠️ `NULL == NULL` is not true, yet `groupBy()` and `dropDuplicates()` treat NULLs as equal — that is why a `null` group appears and dedup collapses NULL rows into one.

In [0]:
# Setup — union ONE row where EVERY column is NULL onto the original 13 rows
null_row_df = spark.createDataFrame([(None, None, None, None, None)], schema=employees_df.schema)

employees_null_row_df = employees_df.union(null_row_df)

employees_null_row_df.sort(col("id").asc_nulls_last()).show()

In [0]:
# Scenario 1 — counts & aggregates: the row COUNTS, but its VALUES don't
# avg = 820000 / 13 (non-NULL salaries), NOT 820000 / 14
employees_null_row_df.select(
    F.count(F.lit(1)).alias("count_all_rows"),
    F.count(col("salary")).alias("count_salary"),
    F.sum(col("salary")).alias("sum_salary"),
    F.avg(col("salary")).alias("avg_salary"),
).show()

In [0]:
# Scenario 2 — groupBy() puts the NULL dept in its OWN group (NULLs group together)
employees_null_row_df.groupBy(col("dept")).count().sort(col("dept").asc_nulls_last()).show()

In [0]:
# Scenario 2 (SQL) — GROUP BY shows the same null group; ORDER BY ... NULLS LAST keeps it visible
employees_null_row_df.createOrReplaceTempView("employees_null_row")

spark.sql("""
    SELECT dept, COUNT(*) AS n
    FROM employees_null_row
    GROUP BY dept
    ORDER BY dept NULLS LAST
""").show()

In [0]:
# Scenario 3 — countDistinct EXCLUDES the NULL dept; plain count includes the NULL row's presence
employees_null_row_df.agg(
    F.count(col("dept")).alias("count_dept"),
    F.countDistinct(col("dept")).alias("count_distinct_dept"),
).show()

In [0]:
# Scenario 4 — NULLs count as EQUAL in dropDuplicates (one NULL row survives); na.drop(how='all') removes fully-NULL rows
print("dropDuplicates(['id']) — NULLs equal, one NULL row survives:", employees_null_row_df.dropDuplicates(["id"]).count())
print("na.drop(how='all')    — fully-NULL row removed           :", employees_null_row_df.na.drop(how="all").count())

## Whole-DataFrame Aggregations — no groupBy()

---

These collapse the ENTIRE DataFrame into ONE summary row. `count()` is the odd one out — it is an ACTION returning a number, not a transformation.
SQL: `SELECT SUM(salary), AVG(salary) ... FROM employees`.

In [0]:
# sum() — total salary across ALL rows (duplicates included; see the gotcha section)
employees_df.select(F.sum(col("salary")).alias("total_salary")).show()

In [0]:
# avg() — average salary across ALL rows
employees_df.select(F.avg(col("salary")).alias("avg_salary")).show()

In [0]:
# min() — lowest salary
employees_df.select(F.min(col("salary")).alias("min_salary")).show()

In [0]:
# max() — highest salary
employees_df.select(F.max(col("salary")).alias("max_salary")).show()

In [0]:
# Spark SQL Equivalent — all summary aggregates in one SELECT
employees_df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT COUNT(*)    AS row_count,
           SUM(salary) AS total_salary,
           AVG(salary) AS avg_salary,
           MIN(salary) AS min_salary,
           MAX(salary) AS max_salary
    FROM employees
""").show()

## `groupBy()` — per-group aggregation

---

`groupBy()` splits rows into groups; every aggregate then runs PER GROUP instead of once. `agg()` is the flexible form — multiple measures, each with `.alias()` for a clean column name.
SQL: `GROUP BY dept`.

In [0]:
# Rows per department — groupBy() then count()
employees_df.groupBy(col("dept")).count().sort(col("dept")).show()

In [0]:
# agg() — multiple named aggregations in ONE call, each aliased
(
    employees_df
    .groupBy(col("dept"))
    .agg(
        F.count(col("id")).alias("emp_count"),
        F.sum(col("salary")).alias("total_salary"),
        F.avg(col("salary")).alias("avg_salary"),
        F.min(col("salary")).alias("min_salary"),
        F.max(col("salary")).alias("max_salary"),
    )
).show()

In [0]:
# Spark SQL Equivalent — GROUP BY with multiple aggregates
spark.sql("""
    SELECT dept,
           COUNT(*)    AS emp_count,
           SUM(salary) AS total_salary,
           AVG(salary) AS avg_salary,
           MIN(salary) AS min_salary,
           MAX(salary) AS max_salary
    FROM employees
    GROUP BY dept
    ORDER BY dept
""").show()

In [0]:
# groupBy on multiple columns — one row per (dept, mgr_id) combination
employees_df.groupBy(col("dept"), col("mgr_id")).count().sort(col("dept"), col("mgr_id")).show()

In [0]:
# countDistinct — UNIQUE employee ids per dept (duplicate rows ignored)
(
    employees_df
    .groupBy(col("dept"))
    .agg(F.countDistinct(col("id")).alias("unique_employees"))
    .sort(col("dept"))
).show()

In [0]:
# Spark SQL Equivalent — COUNT(DISTINCT id) per department
spark.sql("""
    SELECT dept, COUNT(DISTINCT id) AS unique_employees
    FROM employees
    GROUP BY dept
    ORDER BY dept
""").show()

## Sorting Aggregations + `HAVING`

---

`sort()` on the aggregated output orders groups by any measure. To FILTER groups (not rows), use `filter()` AFTER `groupBy()` — that is SQL's `HAVING`; `filter()` BEFORE `groupBy()` is SQL's `WHERE`.

In [0]:
# sort() after agg() — departments by total salary, highest first
(
    employees_df
    .groupBy(col("dept"))
    .agg(F.sum(col("salary")).alias("total_salary"))
    .sort(col("total_salary").desc())
).show()

In [0]:
# DataFrame API equivalent of HAVING — filter() AFTER groupBy filters GROUPS
(
    employees_df
    .groupBy(col("dept"))
    .agg(F.sum(col("salary")).alias("total_salary"))
    .filter(col("total_salary") > 200000)
    .sort(col("total_salary").desc())
).show()

In [0]:
# Spark SQL Equivalent — HAVING filters groups (after aggregation); WHERE cannot
spark.sql("""
    SELECT dept, SUM(salary) AS total_salary
    FROM employees
    GROUP BY dept
    HAVING SUM(salary) > 200000
    ORDER BY total_salary DESC
""").show()

## ⚠️ Duplicates Gotcha — aggregate before vs after dedup

---

The dataset carries 4 duplicate rows on purpose. `sum()` (and `avg()`) count EVERY row — dedup FIRST when duplicates are not real observations.

In [0]:
# sum() double-counts duplicate rows — compare before and after dropDuplicates
total_with_dups = employees_df.agg(F.sum(col("salary")).alias("total")).first()["total"]
total_dedup = employees_df.dropDuplicates(["id"]).agg(F.sum(col("salary")).alias("total")).first()["total"]

print("SUM with duplicates     :", total_with_dups)
print("SUM after dropDuplicates:", total_dedup)
print("Double-counted amount   :", total_with_dups - total_dedup)

## Quick Revision — Aggregation Functions

---

| Method | Scope | Notes |
| --- | --- | --- |
| `df.count()` | whole DF | ACTION — returns a number, not a DataFrame |
| `F.count(col("salary"))` | whole DF | TRANSFORMATION — aggregate column; SKIPS NULLs (= SQL `COUNT(col)`) |
| `F.count(lit(1))` / `F.count("*")` | whole DF | = SQL `COUNT(*)` — every row, NULLs included |
| `sum()` / `avg()` / `min()` / `max()` | whole DF | All SKIP NULLs — `avg()` divides by the NON-NULL count (675000/10) |
| all-NULL column | — | sum/avg/min/max → NULL; only `count()` returns a number (0) |
| `F.coalesce` / SQL `ifnull` / `nvl` | NULLs | coalesce = FIRST non-NULL; ifnull/nvl = 2-arg coalesce (SQL only) |
| one all-NULL row | — | `COUNT(*)` counts it; `groupBy` → own null group; `countDistinct` excludes it; `na.drop(how="all")` removes it |
| `df.select(F.sum(col("salary")))` | whole DF | One summary row |
| `df.groupBy(col("dept")).count()` | per group | Adds a `count` COLUMN per group |
| `df.groupBy(...).agg(...)` | per group | Multiple measures, each with `.alias()` |
| `F.countDistinct(col("id"))` | per group | UNIQUE values — ignores duplicate rows |
| `filter()` AFTER `groupBy()` | groups | = SQL `HAVING` (WHERE filters rows, before) |
| `sort()` after `agg()` | groups | Order groups by any measure |
| ⚠️ `sum()` over duplicates | whole DF | Duplicates are double-counted — dedup first |

> [!TIP]
> 💡 `agg()` with `.alias()` per measure beats chained `.sum()`/`.avg()` — clean column names and multiple aggregates in one pass.

> [!WARNING]
> ⚠️ `HAVING` filters GROUPS (after aggregation) — `WHERE` filters ROWS (before). In the DF API: `filter()` after `groupBy()` = HAVING; before = WHERE.

# Quick Revision Notes (Handwritten-Style) — AGGREGATION FUNCTIONS

---

## 1 — Syntax map

* Whole DF: `df.select(F.sum(col("salary")))` → ONE summary row
* Per group: `df.groupBy(col("dept")).agg(...)`
* SQL: `SELECT SUM(salary) FROM emp` / `... GROUP BY dept`
* `count()` is an ACTION (returns a number); sum/avg/min/max are transformations

## 2 — count() vs countDistinct()

* `df.count()` — rows INCLUDING duplicates
* `F.count(col("id"))` — non-NULL values only
* `F.countDistinct(col("id"))` — UNIQUE ids only; snake_case alias: `F.count_distinct(col("id"))`
* `F.sumDistinct(...)` == `F.sum_distinct(...)` — sum of UNIQUE values; same alias rule
* SQL: `COUNT(*)` counts rows; `COUNT(col)` skips NULLs
* `F.count(lit(1))` == `COUNT(*)` — every row, NULLs included (13 vs 10 here)
* sum/avg/min/max SKIP NULLs — avg = sum ÷ NON-NULL count (675000/10 = 67500)
* ALL-NULL column: sum/avg/min/max → NULL; count → 0

## 3 — agg() is the flexible one

```python
df.groupBy(col("dept")).agg(
    F.count(col("id")).alias("emp_count"),
    F.sum(col("salary")).alias("total_salary"),
    F.avg(col("salary")).alias("avg_salary"),
)
```

## 4 — Sorting + filtering aggregations ⚠️

* `sort(col("total_salary").desc())` AFTER agg — ordered groups
* `filter()` AFTER groupBy = SQL HAVING (filters groups)
* `filter()` BEFORE groupBy = SQL WHERE (filters rows)

## 5 — Duplicates gotcha ⚠️

* `sum()` double-counts duplicate rows (820000 vs 560000 here)
* Dedup FIRST when duplicates are not real observations:
  `df.dropDuplicates(["id"]).agg(F.sum(col("salary")))`

## 6 — Handling NULLs ⚠️

* `coalesce(c1, c2, ...)` — FIRST non-NULL (DF + SQL) — the DEFAULT choice
* SQL only: `IFNULL(c, d)` == `NVL(c, d)` == 2-arg `coalesce` — same performance; use for a single SQL fallback, pick one and stay consistent
* `NULLIF(a, b)` — NULL when a == b (DF: `when`/`otherwise`)
* `isNull()` / `IS NULL` — NEVER use `== NULL`
* `df.na.fill({...})` — replace; `df.na.drop(subset=[...])` — remove (13 → 10)
* all-NULL row: `COUNT(*)` counts it (14); value aggregates skip; `groupBy` → OWN null group; `countDistinct` excludes; `dropDuplicates` keeps ONE; `na.drop(how="all")` removes it

## 7 — Other built-in aggregates (full table at the top of the notebook)

* `first()` / `last()` — first/last value in group; default = RESPECT NULLS, `ignorenulls=True` = IGNORE NULLS (first/last NON-NULL)
* `collect_list()` — array WITH duplicates; `collect_set()` — array of UNIQUE values
* `stddev` / `variance` (+ `_pop` / `_samp`) — sample vs population flavours
* `skewness` / `kurtosis` — shape of the distribution (asymmetry / heavy tails)
* `corr(c1, c2)` — Pearson correlation (−1 to +1); `covar_pop` / `covar_samp` — covariance
* `approx_count_distinct` — fast HyperLogLog++ distinct count; `percentile_approx(col, p)` — approximate percentile (median = 0.5)
* SQL only: `MEDIAN` (exact p50), `MODE`, `ANY_VALUE`, `EVERY/SOME/ANY`, `BIT_AND/OR/XOR`

## Quick Recall Box ⭐

* count = action; sum/avg/min/max = transformation via select()/agg()
* groupBy().count() adds a count COLUMN — df.count() returns a NUMBER
* agg() + alias() for clean names and multiple measures in one pass
* countDistinct = unique count; count = row count
* snake_case aliases: `count_distinct` == `countDistinct`, `sum_distinct` == `sumDistinct` — same functions, newer PySpark style
* filter after groupBy = HAVING, not WHERE
* Sort aggregated output: sort(col("alias").desc())
* Duplicates inflate sum/avg — dedup first if needed
* COUNT(*) counts rows; COUNT(col) skips NULLs
* sum/avg/min/max skip NULLs too — avg divides by NON-NULL count only
* coalesce = first non-NULL; IFNULL/NVL = 2-arg coalesce (SQL only, same performance)
* all-NULL row: counts in COUNT(*), groups under NULL, vanishes from countDistinct
* first/last default = RESPECT NULLS; ignorenulls=True = IGNORE NULLS
* collect_list keeps duplicates; collect_set drops them